In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma  # Can also use Pinecone, Weaviate, Qdrant, Milvus, or FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.callbacks import StreamingStdOutCallbackHandler
try:
    from langchain_classic.chains import RetrievalQA  # langchain>=1.0
except ImportError:
    from langchain.chains import RetrievalQA  # langchain<1.0
import os



C:\Users\goran\AppData\Local\Temp\ipykernel_25356\3329287528.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader


In [2]:
class RAGSystem:
    def __init__(self, persist_directory="./rag_db"):
        self.persist_directory = persist_directory
        self.embeddings = GoogleGenerativeAIEmbeddings(
            model="models/gemini-embedding-001"
        )
        self.vector_store = None
        self.retriever = None
        
    def load_documents(self, data_path):
        """Load documents from various sources"""
        documents = []
        
        # Load PDFs
        pdf_loader = DirectoryLoader(
            data_path,
            glob="**/*.pdf",
            loader_cls=PyPDFLoader,
            show_progress=True
        )
        documents.extend(pdf_loader.load())
        
        # Load text files
        text_loader = DirectoryLoader(
            data_path,
            glob="**/*.txt",
            loader_cls=TextLoader,
            show_progress=True
        )
        documents.extend(text_loader.load())
        
        print(f"Loaded {len(documents)} documents")
        return documents
    
    def chunk_documents(self, documents, chunk_size=1000, chunk_overlap=200):
        """Split documents into optimal chunks"""
        # Create text splitter with semantic awareness
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
            separators=[
                "\n\n",    # Paragraph breaks
                "\n",       # Line breaks
                ". ",       # Sentence ends
                ", ",       # Clause breaks
                " ",        # Word breaks
                ""          # Character breaks
            ]
        )
        
        # Split documents
        chunks = text_splitter.split_documents(documents)
        
        # Add metadata to chunks
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = i
            chunk.metadata["chunk_size"] = len(chunk.page_content)
            
        print(f"Created {len(chunks)} chunks")
        return chunks

    def create_vector_store(self, chunks):
        """Create and persist vector store"""
        # Create vector store from chunks
        # Can also use Pinecone, Weaviate, Qdrant, Milvus, or FAISS
        self.vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            persist_directory=self.persist_directory,
            collection_metadata={"hnsw:space": "cosine"}
        )
        
        # Persist to disk
        # Note: Chroma 0.4.x+ automatically persists documents
        print(f"Vector store created with {len(chunks)} chunks")

    def load_vector_store(self):
        """Load existing vector store"""
        self.vector_store = Chroma(
            persist_directory=self.persist_directory,
            embedding_function=self.embeddings
        )
        print("Vector store loaded from disk")

    def setup_retriever(self, k=5, search_type="similarity", score_threshold=0.5):
        """Configure retriever with advanced options"""
        if not self.vector_store:
            raise ValueError("Vector store not initialized")
        
        # Each search type only accepts its own kwargs; Chroma.query() rejects the others
        search_kwargs = {"k": k}
        if search_type == "similarity_score_threshold":
            search_kwargs["score_threshold"] = score_threshold
        elif search_type == "mmr":
            search_kwargs["fetch_k"] = k * 2
        
        self.retriever = self.vector_store.as_retriever(
            search_type=search_type,
            search_kwargs=search_kwargs
        )
        
        return self.retriever

    def create_qa_chain(self, streaming=True, model="gemini-flash-latest"):
        """Create the complete RAG chain"""
        # "-latest" alias avoids 404s when a pinned model version is retired
        llm = ChatGoogleGenerativeAI(
            model=model,
            temperature=0.3,
            streaming=streaming,
            callbacks=[StreamingStdOutCallbackHandler()] if streaming else []
        )
        
        # Create custom prompt template
        prompt_template = """You are a helpful AI assistant. Use the following context to answer the question.
If you don't know the answer based on the context, say "I don't have enough information to answer this question."

Context:
{context}

Question: {question}

Answer: """
        
        PROMPT = PromptTemplate(
            template=prompt_template,
            input_variables=["context", "question"]
        )
        
        # Create retrieval chain
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type="stuff",  # Use "map_reduce" for long contexts
            retriever=self.retriever,
            return_source_documents=True,
            chain_type_kwargs={"prompt": PROMPT}
        )
        
        return qa_chain

    def query(self, question, verbose=False):
        """Query the RAG system"""
        if not self.retriever:
            raise ValueError("Retriever not initialized")
        
        # Create QA chain
        qa_chain = self.create_qa_chain()
        
        # Execute query
        result = qa_chain.invoke({"query": question})
        
        # Format response
        response = {
            "answer": result["result"],
            "source_documents": []
        }
        
        # Add source information
        for doc in result.get("source_documents", []):
            source_info = {
                "content": doc.page_content[:200] + "...",
                "metadata": doc.metadata,
                "similarity_score": getattr(doc, "score", None)
            }
            response["source_documents"].append(source_info)
        
        if verbose:
            print(f"\nRetrieved {len(response['source_documents'])} documents")
            
        return response


In [3]:
# Initialize RAG system - works with ChromaDB, Pinecone, Weaviate, Qdrant, or Milvus
rag = RAGSystem(persist_directory="./my_rag_db")

In [4]:
# Create sample documents for testing (replace with your own data)
if not os.path.exists("./data"):
    os.makedirs("./data")
    
    # Create sample text file
    with open("./data/sample.txt", "w", encoding="utf-8") as f:
        f.write("""RAG (Retrieval Augmented Generation) Systems

RAG systems combine the power of retrieval and generation to provide accurate, contextual responses.

Key Benefits:
1. Reduces hallucinations in AI responses
2. Enables AI to access up-to-date information
3. Provides source attribution for answers
4. Allows domain-specific knowledge integration

How RAG Works:
- Documents are processed and stored in a vector database
- User queries are converted to embeddings
- Similar document chunks are retrieved
- Retrieved context is used to generate accurate responses

Popular vector databases for RAG include ChromaDB, Pinecone, Weaviate, and FAISS.""")

In [5]:
# Option 1: Build new RAG index from scratch (for beginners)
try:
    documents = rag.load_documents("./data")
    if documents:
        print(f"Loaded {len(documents)} documents")
        chunks = rag.chunk_documents(documents, chunk_size=1000, chunk_overlap=200)
        print(f"Created {len(chunks)} chunks")
        rag.create_vector_store(chunks)  # Creates embeddings and stores in vector database
        print("Vector store created successfully!")
        
        # Setup retriever
        rag.setup_retriever(k=5, search_type="similarity")
        
        # How to query your RAG system - simple example
        question = "What are the main benefits of using RAG systems?"
        response = rag.query(question, verbose=True)  # Returns answer + sources
        
        print(f"\nAnswer: {response['answer']}")
        print(f"\nSources used: {len(response['source_documents'])}")
        
        # Show source content
        for i, doc in enumerate(response['source_documents']):
            print(f"\nSource {i+1}: {doc['content']}")
    else:
        print("No documents found in ./data directory")
        
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you have documents in ./data directory and GOOGLE_API_KEY is set")


0it [00:00, ?it/s]
0it [00:00, ?it/s]

Loaded 0 documents
No documents found in ./data directory
